# 张量

张量是一种特殊的数学结构，与数组和矩阵非常相似。在 PyTorch 中，我们使用张量来编码模型的输入、输出以及模型的参数。

张量与 NumPy 的 ndarray 非常相似，不同之处在于张量可以在 GPU 或其他硬件加速器上运行。事实上，张量和 NumPy 数组通常可以共享底层的内存，从而无需复制数据。张量还针对自动求导进行了优化。如果你熟悉 ndarray，那么你会对 Tensor API 感到非常亲切。

In [ ]:
import torch
import numpy as np

## 初始化张量

可以通过多种方式初始化张量。请看以下示例：

**直接从数据创建**

可以直接从数据创建张量。数据类型会自动推断。

In [ ]:
import torch
import numpy as np

data = [[1, 2],[3, 4]]
x_data = torch.tensor(data)
print(x_data)

**从 NumPy 数组创建**

可以从 NumPy 数组创建张量（反之亦然）。

In [ ]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)
print(x_np)

**从另一个张量创建**

新张量保留了原张量的属性（形状、数据类型），除非显式重写。

In [ ]:
x_ones = torch.ones_like(x_data)  # retains the properties of x_data
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float)  # overrides the datatype of x_data
print(f"Random Tensor: \n {x_rand} \n")

**使用随机或常量值创建**

`shape` 是一个元组，代表张量的维度。在下面的函数中，它决定了输出张量的维度。

In [ ]:
shape = (2, 3)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")

---

## 张量的属性

张量属性描述了其形状、数据类型以及存储它的设备。

In [ ]:
tensor = torch.rand(3, 4)

print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

---

## 张量运算

PyTorch 提供了 1200 多种张量运算，包括算术运算、线性代数、矩阵操作（转置、索引、切片）、采样等。所有这些操作都可以在 CPU 以及 CUDA、MPS、MTIA 或 XPU 等加速器上运行。

默认情况下，张量是在 CPU 上创建的。我们需要在检查加速器可用性后，使用 `.to` 方法将张量显式移动到加速器上。请记住，在不同设备间复制大张量在时间和内存方面可能非常昂贵！

In [ ]:
# We move our tensor to the current accelerator if available
if torch.accelerator.is_available():
    tensor = tensor.to(torch.accelerator.current_accelerator())

**标准的类似 numpy 的索引和切片**

In [ ]:
tensor = torch.ones(4, 4)
print(f"First row: {tensor[0]}")
print(f"First column: {tensor[:, 0]}")
print(f"Last column: {tensor[..., -1]}")
tensor[:, 1] = 0
print(tensor)

**拼接张量** ：你可以使用 `torch.cat` 沿指定维度连接一系列张量。另请参阅 `torch.stack`，这是另一种与 `torch.cat` 略有不同的张量拼接运算符。

In [ ]:
t1 = torch.cat([tensor, tensor, tensor], dim=1)
print(t1)

**算术运算**

In [ ]:
# 矩阵乘法（三种等价写法）
y1 = tensor @ tensor.T
y2 = tensor.matmul(tensor.T)

y3 = torch.rand_like(y1)
torch.matmul(tensor, tensor.T, out=y3)

# 逐元素乘法（三种等价写法）
z1 = tensor * tensor
z2 = tensor.mul(tensor)

z3 = torch.rand_like(tensor)
torch.mul(tensor, tensor, out=z3)

**单元素张量** ：如果你有一个单元素张量（例如通过将张量的所有值聚合为一个值得到），你可以使用 `item()` 将其转换为 Python 数值。

In [ ]:
agg = tensor.sum()
agg_item = agg.item()
print(agg_item, type(agg_item))

**原地操作 (In-place operations)** ：将结果存储在操作数中的操作称为“原地”操作。它们以 `_` 后缀表示。例如： `x.copy_(y)` 或 `x.t_()` 将会改变 `x` 。

> **注意**：原地操作可以节省一些内存，但在计算导数时可能会因为立即丢失计算历史而产生问题。因此，通常不鼓励使用它们。

In [ ]:
print(f"{tensor} \n")
tensor.add_(5)
print(tensor)

---

## 与 NumPy 的交互

CPU 上的张量和 NumPy 数组可以共享底层内存位置，改变其中一个也会改变另一个。

### 张量转 NumPy 数组

In [ ]:
t = torch.ones(5)
print(f"t: {t}")
n = t.numpy()
print(f"n: {n}")

张量的变化会反映在 NumPy 数组中。

In [ ]:
t.add_(1)
print(f"t: {t}")
print(f"n: {n}")

### NumPy 数组转张量

In [ ]:
n = np.ones(5)
t = torch.from_numpy(n)

NumPy 数组的变化会反映在张量中。

In [ ]:
np.add(n, 1, out=n)
print(f"t: {t}")
print(f"n: {n}")